<a href="https://colab.research.google.com/github/sarans2223/NUMPY/blob/main/KMEANS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("carrie1/ecommerce-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ecommerce-data' dataset.
Path to dataset files: /kaggle/input/ecommerce-data


In [8]:
import os
csv_path = os.path.join(path, "data.csv")
df = pd.read_csv(csv_path, encoding="ISO-8859-1")

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [9]:
df = pd.read_csv(csv_path, encoding="ISO-8859-1")

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [12]:
df.shape

(541909, 8)

In [13]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
country_counts = df["Country"].value_counts().head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=country_counts.values, y=country_counts.index)
plt.title("Top 10 Countries by Number of Transactions")
plt.xlabel("Number of Transactions")
plt.ylabel("Country")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["Quantity"], bins=50)
plt.title("Quantity Distribution")
plt.xlabel("Quantity")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["UnitPrice"], bins=50)
plt.title("Unit Price Distribution")
plt.xlabel("Unit Price")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [ ]:
df = df.dropna(subset=["CustomerID"])

In [ ]:
df = df[df["Quantity"] > 0]

In [ ]:
df = df[df["UnitPrice"] > 0]

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("Rows after cleaning:", len(df))
print("Customers:", df["CustomerID"].nunique())

In [ ]:
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

df.head()

In [ ]:
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

print("Reference Date:", reference_date)

In [ ]:
rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (reference_date - x.max()).days,
    "InvoiceNo": "nunique",
    "TotalAmount": "sum"
})

rfm.columns = ["Recency", "Frequency", "Monetary"]

rfm.head()

In [ ]:
rfm.describe()

In [ ]:
rfm.describe()

In [ ]:
plt.figure(figsize=(15,4))

plt.subplot(1,3,1)
sns.histplot(rfm["Recency"], bins=30)
plt.title("Recency Distribution")

plt.subplot(1,3,2)
sns.histplot(rfm["Frequency"], bins=30)
plt.title("Frequency Distribution")

plt.subplot(1,3,3)
sns.histplot(rfm["Monetary"], bins=30)
plt.title("Monetary Distribution")

plt.tight_layout()
plt.show()

In [ ]:
rfm_log = rfm.copy()

rfm_log["Recency"] = np.log1p(rfm_log["Recency"])
rfm_log["Frequency"] = np.log1p(rfm_log["Frequency"])
rfm_log["Monetary"] = np.log1p(rfm_log["Monetary"])

rfm_log.head()

In [ ]:
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm_log)

rfm_scaled = pd.DataFrame(
    rfm_scaled,
    columns=["Recency", "Frequency", "Monetary"],
    index=rfm.index
)

rfm_scaled.head()

In [ ]:
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm_log)

rfm_scaled = pd.DataFrame(
    rfm_scaled,
    columns=["Recency", "Frequency", "Monetary"],
    index=rfm.index
)

rfm_scaled.head()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(K, inertia, marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")

plt.show()

In [ ]:
silhouette_scores = []

for k in range(2, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(rfm_scaled)

    score = silhouette_score(rfm_scaled, labels)
    silhouette_scores.append(score)

    print(f"K = {k}, Silhouette Score = {score:.4f}")

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    range(2,11),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")

plt.show()

In [ ]:
optimal_k = 4

In [ ]:
kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

rfm.head()

In [ ]:
final_silhouette = silhouette_score(
    rfm_scaled,
    rfm["Cluster"]
)

print("Final Silhouette Score:", round(final_silhouette, 4))

In [ ]:
cluster_counts = rfm["Cluster"].value_counts().sort_index()

print(cluster_counts)

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    x=cluster_counts.index,
    y=cluster_counts.values
)

plt.title("Number of Customers in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
cluster_profile = rfm.groupby("Cluster").agg({
    "Recency": "mean",
    "Frequency": "mean",
    "Monetary": "mean",
    "Cluster": "count"
})

cluster_profile.rename(
    columns={"Cluster": "CustomerCount"},
    inplace=True
)

cluster_profile

In [ ]:
cluster_profile.sort_values(
    by="Monetary",
    ascending=False
)

In [ ]:
cluster_profile["Percentage"] = (
    cluster_profile["CustomerCount"] /
    cluster_profile["CustomerCount"].sum()
) * 100

cluster_profile

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    x=cluster_profile.index,
    y=cluster_profile["Recency"]
)

plt.title("Average Recency by Cluster")
plt.xlabel("Cluster")
plt.ylabel("Average Recency (Days)")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    x=cluster_profile.index,
    y=cluster_profile["Frequency"]
)

plt.title("Average Frequency by Cluster")
plt.xlabel("Cluster")
plt.ylabel("Average Number of Orders")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    x=cluster_profile.index,
    y=cluster_profile["Monetary"]
)

plt.title("Average Monetary Value by Cluster")
plt.xlabel("Cluster")
plt.ylabel("Average Spending")

plt.show()

In [ ]:
heatmap_data = cluster_profile[
    ["Recency", "Frequency", "Monetary"]
].copy()

# Normalize each column for comparison
heatmap_data = (
    heatmap_data - heatmap_data.mean()
) / heatmap_data.std()

plt.figure(figsize=(8,5))

sns.heatmap(
    heatmap_data,
    annot=True,
    cmap="coolwarm",
    center=0
)

plt.title("Cluster RFM Profile")
plt.show()

In [ ]:
cluster_names = {
    0: "High Value Loyal",
    1: "At Risk",
    2: "New Customers",
    3: "Regular Customers"
}

rfm["Segment"] = rfm["Cluster"].map(cluster_names)

rfm.head()

In [ ]:
final_profile = rfm.groupby("Segment").agg(
    Recency=("Recency", "mean"),
    Frequency=("Frequency", "mean"),
    Monetary=("Monetary", "mean"),
    Customers=("Cluster", "count")
)

final_profile

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=rfm,
    x="Frequency",
    y="Monetary",
    hue="Segment",
    palette="Set2",
    s=80
)

plt.title("Customer Segmentation using RFM")
plt.xlabel("Frequency")
plt.ylabel("Monetary Value")

plt.show()

In [ ]:
print("Optimal K:", optimal_k)
print("Final Silhouette Score:", round(final_silhouette, 4))

print("\nCluster Profile:")
display(final_profile)